####Notebook de Entrada

Notebook de entrada principal do algoritmo de reumatologia, os dados sao extraidos das seguintes tabelas:<br>
<br>
`gold_corporativo_ia.corporativo.tb_gold_mov_exame`<br>
`gold_corporativo_ia.corporativo.tb_gold_mov_paciente` 
<br>
<br>Após realizar o filtro dos exames os dados sao enriquecidos com as informações dos pacientes, posterior a vw resultante é usada para inserção dos dados na tabela final.

####Variáveis de Ambiente

In [0]:
dbutils.widgets.text("environment", "dev", "Environment")

In [0]:
environment = dbutils.widgets.get("environment")

In [0]:
if environment in ["hml", "prd"]:
    environment_tbl = ""
else:
    environment_tbl = f"{environment}_"

In [0]:
OUTPUT_TABLENAME = f"{environment_tbl}tb_diamond_mod_reumatologia"
VW_BALIZADORA = f"{environment_tbl}vw_reumatologia_balizador"
SCHEMA = "ia"

In [0]:
print('Environment:   ', environment_tbl)
print('Tabela destino:', OUTPUT_TABLENAME)
print('VW balizadora: ', VW_BALIZADORA)
print('VW balizadora: ', SCHEMA)

In [0]:
# spark.sql(f"""DROP TABLE IF EXISTS {CATALOG}.{OUTPUT_TABLENAME}""")

####Tabelas & Views

In [0]:
# spark.sql(f"""
# CREATE OR REPLACE TEMP VIEW {VW_BALIZADORA} AS
# WITH base AS (
#   SELECT
#       e.id_unidade                                     AS idunidade,
#       e.nme_hospital                                   AS unidade,
#       p.id_paciente                                    AS id_pct,
#       p.cli_nome                                       AS paciente,
#       p.cli_genero                                     AS sexo,
#       p.cli_data_nascimento                            AS nascimento,
#       p.doc_cpf                                        AS cpf,
#       p.cli_telefone_numero                            AS telefonePaciente,
#       p.cli_telefone_ddd                               AS telefonePacienteDDD,
#       e.tp_procedimento                                AS procedimento,
#       e.num_pedido_integracao                          AS num_pedido_integracao,
#       e.nme_regional_hospital                          AS nme_regional_hospital,
#       e.nme_convenio                                   AS nme_convenio,

#       regexp_replace(lower(coalesce(e.dsc_procedimento_integracao, cast(e.cod_procedimento as string), '')), '[\\u00A0\\u2007\\u202F]', ' ') as exame_nr,

#       /*calculo o numero de meses entre as duas datas e divido por 12 para ter o valor em anos*/
#       FLOOR(months_between(DATE(e.dt_dia_exame), DATE(p.cli_data_nascimento)) / 12) AS idade_anos,
#       CASE
#         WHEN exame_nr RLIKE '(tomografia|angio|tc|tomo)'
#           THEN 'CT'
#         ELSE 'IND'
#       END                                              AS modalidade,
#       translate(
#         regexp_replace(
#           lower(coalesce(exame_nr, cast(e.cod_procedimento as string), '')),
#           '[\\u00A0\\u2007\\u202F]', ' '
#         ),
#         'áàãâäéèêëíìîïóòõôöúùûüç',
#         'aaaaaeeeeiiiiooooouuuuc'
#       ) AS exame,

#       e.dt_dia_exame                                   AS dataexame,
#       translate(
#         regexp_replace(
#           lower(coalesce(e.nme_origem_exame, e.tp_proced_classe_encontro, '')),
#           '[\\u00A0\\u2007\\u202F]', ' '
#         ),
#         'áàãâäéèêëíìîïóòõôöúùûüç',
#         'aaaaaeeeeiiiiooooouuuuc'
#       ) AS tipoexame,

#       CAST(e.id_exame AS STRING)                       AS an,

#       e.dt_liberacao_laudo                             AS DataLaudoLiberado,
#       array_join(
#         transform(e.proced_lista_exames, x -> x.laudo_transformado),
#         '\n'
#       ) AS Laudo,
#       COALESCE(e.dt_liberacao_laudo, e.dt_previsao_laudo ) AS DataLaudoFinal,
#       CASE
#         WHEN e.dt_liberacao_laudo IS NOT NULL THEN 5
#         WHEN e.dt_previsao_laudo  IS NOT NULL THEN 4
#         ELSE 0
#       END AS idstatus

#   FROM gold_corporativo_ia.corporativo.tb_gold_mov_exame   e
#   LEFT JOIN gold_corporativo_ia.corporativo.tb_gold_mov_paciente p
#          ON p.id_paciente = e.id_patient
# ),
# filtrada AS (
#   SELECT *
#   FROM base
#   WHERE
#     UPPER(trim(procedimento)) IN  ('IMG', 'IMA')
#     AND(
#       LOWER(exame_nr) ILIKE '%tomografia' OR LOWER(exame_nr) ILIKE '%ultra%' OR LOWER(exame_nr) ILIKE '%resso%'
#       OR LOWER(exame_nr) ILIKE '%tc' OR LOWER(exame_nr) ILIKE '%usg' OR LOWER(exame_nr) ILIKE '%rm'
#     )
#     AND(
#       LOWER(exame_nr) ILIKE '%crânio%' or LOWER(exame_nr) ILIKE '%cranio%' or LOWER(exame_nr) ILIKE '%joelho%' or LOWER(exame_nr) ILIKE '%articula%' or LOWER(exame_nr) ILIKE '%temporomandibular%' or LOWER(exame_nr) ILIKE '%face%' or LOWER(exame_nr) ILIKE '%coluna%' or LOWER(exame_nr) ILIKE '%lombar%' or LOWER(exame_nr) ILIKE '%tornozelo%' or LOWER(exame_nr) ILIKE '%pé' or LOWER(exame_nr) ILIKE '%pe' or LOWER(exame_nr) ILIKE '%mao%' or LOWER(exame_nr) ILIKE '%mão%' or LOWER(exame_nr) ILIKE '%ombro%' or LOWER(exame_nr) ILIKE '%punho%' or LOWER(exame_nr) ILIKE '%cervical%' or LOWER(exame_nr) ILIKE '%quadril%' or LOWER(exame_nr) ILIKE '%quadris%' or LOWER(exame_nr) ILIKE '%bacia%' or LOWER(exame_nr) ILIKE '%perna%' or LOWER(exame_nr) ILIKE '%braco%' or LOWER(exame_nr) ILIKE '%braço%' or LOWER(exame_nr) ILIKE '%cotovelo%' or LOWER(exame_nr) ILIKE '%coxa%' or LOWER(exame_nr) ILIKE '%intracranian%' or LOWER(exame_nr) ILIKE '%pescoço%' or LOWER(exame_nr) ILIKE '%pescoco%' or LOWER(exame_nr) ILIKE '%cabeça%' or LOWER(exame_nr) ILIKE '%cabeca%' or LOWER(exame_nr) ILIKE '%dorsal%' or LOWER(exame_nr) ILIKE '%osso%' or LOWER(exame_nr) ILIKE '%vertebra%' or LOWER(exame_nr) ILIKE '%lombossacr%' or LOWER(exame_nr) ILIKE '%mastoid%' or LOWER(exame_nr) ILIKE '%sacro%' or LOWER(exame_nr) ILIKE '%sacra%'
#     )
#     AND(
#       LOWER(exame_nr) NOT LIKE '%angio%' AND LOWER(exame_nr) NOT LIKE '%artérias%' AND LOWER(exame_nr) NOT LIKE '%carótidas%' AND LOWER(exame_nr) NOT LIKE '%arterias%' AND LOWER(exame_nr) NOT LIKE '%carotidas%' AND LOWER(exame_nr) NOT LIKE '%venos%' AND LOWER(exame_nr) NOT LIKE '%paaf%' AND LOWER(exame_nr) NOT LIKE '%punção%'AND LOWER(exame_nr) NOT LIKE '%biop%'AND LOWER(exame_nr) NOT LIKE '%bióp%'
#     )  
#     AND DataLaudoFinal IS NOT NULL
#     AND exame != ""
#     AND DATE(DataLaudoFinal) BETWEEN date_sub(current_date(), 30)
#                                  AND date_sub(current_date(), 1) -- ontem
# ),

# dedup AS (
#   SELECT
#     f.*,
#     ROW_NUMBER() OVER (
#       PARTITION BY an
#       ORDER BY TIMESTAMP(DataLaudoFinal) DESC
#     ) AS rn,
#     CASE WHEN ROW_NUMBER() OVER (
#              PARTITION BY an
#              ORDER BY TIMESTAMP(DataLaudoFinal) DESC
#          ) = 1
#          THEN 1 ELSE 0 END AS an_duplicado
#   FROM filtrada f
# )
# SELECT
#   idunidade,
#   unidade,
#   id_pct,
#   paciente,
#   telefonePacienteDDD,
#   telefonePaciente,
#   sexo,
#   nascimento,
#   idade_anos,
#   cpf,
#   modalidade,
#   exame,
#   num_pedido_integracao,
#   nme_regional_hospital,
#   nme_convenio,
#   dataexame,
#   tipoexame,
#   an,
#   an_duplicado,
#   idstatus,
#   -- 4 AS idstatus,                 -- mantido
#   -- idLaudo,                       
#   DataLaudoFinal AS DataLaudoLiberado,  -- >>> usa a data escolhida
#   Laudo,
#   -- current_timestamp() as datCarga
#   CAST(from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo') AS TIMESTAMP_NTZ) AS dataExecucaoModelo
# FROM dedup;
# """)

In [0]:
# %sql
# -- CREATE OR REPLACE TABLE ia.tb_diamond_mod_pulmao_filtro AS
# CREATE OR REPLACE TEMP VIEW vw_novos AS
# WITH base AS (
#   SELECT
#       e.id_unidade                                     AS idunidade,
#       e.nme_hospital                                   AS unidade,
#       p.id_paciente                                    AS id_pct,
#       p.cli_nome                                       AS paciente,
#       p.cli_genero                                     AS sexo,
#       p.cli_data_nascimento                            AS nascimento,
#       p.doc_cpf                                        AS cpf,
#       e.tp_procedimento                                AS procedimento,
#       /*para aplicar os filtros em "filtradas" aplico lower, evito nulos e trato NBSP*/
#       /*NBSP nao e igual espaco comum, pode quebrar like, rlike - comum em texto html*/
#       -- regexp_replace(lower(coalesce(e.dsc_procedimento_integracao,'')), '\u00A0', ' ') as exame_nr,
#       -- regexp_replace(lower(coalesce(e.dsc_procedimento_integracao,'')), '[\\u00A0\\u2007\\u202F]', ' ') as exame_nr,
#       regexp_replace(lower(coalesce(e.dsc_procedimento_integracao, cast(e.cod_procedimento as string), '')), '[\\u00A0\\u2007\\u202F]', ' ') as exame_nr,

#       /*calculo o numero de meses entre as duas datas e divido por 12 para ter o valor em anos*/
#       FLOOR(months_between(DATE(e.dt_dia_exame), DATE(p.cli_data_nascimento)) / 12) AS idade_anos,

#       /*100% das obs do IDOR constam como CT que seria a modalidade do exame entao mantive*/
#       /*Me parece se tratar apenas de uma flag, podendo ser alterado para futuras capturas*/
#       /*que nao sejam esta categoria atual, que seja por exemplo TGP-(TG), USG-(US) etc*/
#       /*exemplo desta formatacao esta no ntb montaRedcap_fase2 na func tipoModadlidade()*/
#       /*pode ser o campo tp_procedimento da tbl exame sendo IMG*/
#       CASE
#         -- WHEN LOWER(COALESCE(e.dsc_procedimento_integracao, '')) RLIKE '(tomografia|angio)'
#         WHEN exame_nr RLIKE '(tomografia|angio|tc|tomo)'
#           THEN 'CT'
#         ELSE 'IND'
#       END                                              AS modalidade,

#       /*pego a col dsc_procedimento_integracao quando nao for null, senão uso cod_procedimento*/
#       /*dsc_procedimento...: score de cálcio - TOMOGRAFIA COMPUTADORIZADA DO TÓRAX*/
#       /*cod_procedimento: torax*/
#       /*a col dsc é mais completa enquanto a col cod é mais resumida*/
#       -- COALESCE(e.dsc_procedimento_integracao, CAST(e.cod_procedimento AS STRING)) AS exame,
#       -- COALESCE(exame_nr, CAST(e.cod_procedimento AS STRING)) AS exame,
#       translate(
#         regexp_replace(
#           lower(coalesce(exame_nr, cast(e.cod_procedimento as string), '')),
#           '[\\u00A0\\u2007\\u202F]', ' '
#         ),
#         'áàãâäéèêëíìîïóòõôöúùûüç',
#         'aaaaaeeeeiiiiooooouuuuc'
#       ) AS exame,

#       /*pego a data do exame*/
#       e.dt_dia_exame                                   AS dataexame,

#       /*pego a col nme_origem_exame quando nao for null, senão uso tp_proced_clase_encontro*/
#       /*essa col tras informacoes como: Pronto Socorro, Internação, Externo*/
#       /*essa informacao é convertida em codigo esta no ntb montaRedcap_fase2 na func tipoExame()*/
#       -- COALESCE(e.nme_origem_exame, e.tp_proced_classe_encontro) AS tipoexame,
#       translate(
#         regexp_replace(
#           lower(coalesce(e.nme_origem_exame, e.tp_proced_classe_encontro, '')),
#           '[\\u00A0\\u2007\\u202F]', ' '
#         ),
#         'áàãâäéèêëíìîïóòõôöúùûüç',
#         'aaaaaeeeeiiiiooooouuuuc'
#       ) AS tipoexame,

#       /*avaliando as col do exemplo no csv essa é a col que mais de adequa, ela leva o cod de*/
#       /*identificação do exame*/
#       CAST(e.id_exame AS STRING)                       AS an,

#       /* data de liberação do laudo */
#       e.dt_liberacao_laudo                             AS DataLaudoLiberado,

#       /* concatenação dos laudos, estao dentro de um array de struct ignorando nulos */
#       -- CASE
#       --   WHEN e.proced_lista_exames IS NOT NULL THEN
#       --     array_join(
#       --       transform(
#       --         filter(e.proced_lista_exames, x -> x.laudo_transformado IS NOT NULL AND trim(x.laudo_transformado) <> ''),
#       --         x -> x.laudo_transformado
#       --       ),
#       --       '\n'
#       --     )
#       --   ELSE NULL
#       -- END AS Laudo,
#       /*captura independente se null*/
#       array_join(
#         transform(e.proced_lista_exames, x -> x.laudo_transformado),
#         '\n'
#       ) AS Laudo,

#       /*pego a col dt_liberacao_laudo quando nao for null, senão uso dt_previsao_laudo*/
#       /*se tenho data liberação aplico 5, se tenho data previsao aplico 4*/
#       COALESCE(e.dt_liberacao_laudo, e.dt_previsao_laudo ) AS DataLaudoFinal,
#       CASE
#         WHEN e.dt_liberacao_laudo IS NOT NULL THEN 5
#         WHEN e.dt_previsao_laudo  IS NOT NULL THEN 4
#         ELSE 0
#       END AS idstatus

#   /*tabelas exame x pacientes*/
#   FROM gold_corporativo_ia.corporativo.tb_gold_mov_exame   e
#   LEFT JOIN gold_corporativo_ia.corporativo.tb_gold_mov_paciente p
#          ON p.id_paciente = e.id_patient
# ),
# filtrada AS (
#   SELECT *
#   FROM base
#   WHERE
#     -- UPPER(modalidade) = 'CT'
#     -- UPPER(procedimento) = 'IMG'
#     UPPER(trim(procedimento)) IN  ('IMG', 'IMA')
#     -- olhando a exame_nr (dsc_procedimento_integracao - cod_procedimento) titulo/descricao do exame
#     -- aplico o filtro nos exames de interesse, desconsiderando %angio% > %cran% e %pesco%
#     AND (
#         exame_nr LIKE '%torax%' OR exame_nr LIKE '%tórax%'
#       OR exame_nr LIKE '%coronar%' OR exame_nr LIKE '%calcio%' OR exame_nr LIKE '%cálcio%'
#       OR exame_nr LIKE '%abdom%'
#       OR (
#           exame_nr LIKE '%angio%'
#           AND (
#                 exame_nr LIKE '%torax%' OR exame_nr LIKE '%tórax%'
#             OR  exame_nr LIKE '%pulmonar%'
#             OR  exame_nr LIKE '%aorta torac%'
#             OR  exame_nr LIKE '%toracoabdom%'
#             OR  exame_nr LIKE '%pulm(ã|a)o%'
#             OR  exame_nr LIKE '%pulm(õ|o)es%'
#           )
#           AND exame_nr NOT LIKE '%cran%'
#           -- AND exame_nr NOT LIKE '%pesco%'
#         )
#         )
#     -- se atendido os filtros acima entao olhamos os laudos, é necessario achar oe termos a seguir
#     AND (
#       LOWER(COALESCE(Laudo,'')) RLIKE 'n[óo]dulo?s?'
#       OR LOWER(COALESCE(Laudo,'')) RLIKE 'nodular(?:es)?'
#       OR LOWER(COALESCE(Laudo,'')) RLIKE 'micron[óo]dulo?s?'
#       OR LOWER(COALESCE(Laudo,'')) RLIKE 'noduliforme'
#       OR LOWER(COALESCE(Laudo,'')) LIKE '%n&oacute;dul%'        -- HTML entity
#       OR LOWER(COALESCE(Laudo,'')) RLIKE 'n&oacute;dulo?s?'     -- HTML entity
#     )
#     -- se atendida as duas condicoes acima entao olhamos para os laudos buscando as palavras a seguir
#     -- AND (
#     --   LOWER(COALESCE(Laudo,'')) RLIKE '\\bpulm(ã|a)o\\b'         -- pulmão/pulmao
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE 'pulm(ã|a)o'     
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE '\\bpulm(õ|o)es\\b'     -- pulmões/pulmoes
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE 'pulm(õ|o)es'     
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE '\\bpulmonar\\b'
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE 'pulmonar'
#     --   OR LOWER(COALESCE(Laudo,'')) LIKE '%pulm&atilde;o%'        -- HTML entity
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE '\\bpar[êe]nquima\\s+pulmonar\\b|\\bparenquimapulmonar\\b'
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE '\\blobo\\s+(?:superior|m[é e]dio|medio|inferior)\\b'
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE '\\blobosuperior\\b|\\blobomedio\\b|\\bloboinferior\\b'
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE '\\bl[íi]ngula\\b'     -- língula/lingula
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE '\\bapico\\s+(?:anterior|posterior|lateral|medial)\\b'
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE '\\bapicoposterior|apical\\b'
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE '\\bbasal\\s+(?:anterior|posterior|medial|lateral|anteromedial)?\\b'
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE '\\bbasalanterior\\b|\\bbasalposterior\\b|\\bbasalmedial\\b|\\bbasallateral\\b|\\bbasalanteromedial\\b'
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE '\\bperih?ilar\\b'
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE '\\bparacard[ií]aco\\b'
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE '\\bsubpleural\\b'      -- ainda é parênquima
#     -- )
#     -- AND (
#     --   LOWER(COALESCE(Laudo,'')) RLIKE 'pulm(ã|a)o'         -- pulmão/pulmao
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE 'pulm(õ|o)es'     -- pulmões/pulmoes
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE 'pulmonar'
#     --   OR LOWER(COALESCE(Laudo,'')) LIKE '%pulm&atilde;o%'        -- HTML entity
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE 'pulm&atilde;o'         -- HTML entity 
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE 'par[êe]nquima\\s+pulmonar|parenquimapulmonar'
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE 'lobo\\s+(superior|m[eé]dio|medio|inferior)|lobosuperior|lobomedio|loboinferior'
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE 'língula|lingula|lingu(l|l)a'     -- língula/lingula
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE '(apico|apicoposterior|apical|anterior|posterior|lateral|medial)'
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE 'apico(poste?rior)?|apical|anterior|posterior|lateral|medial'
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE 'basal(\\s+(anterior|posterior|medial|lateral|anteromedial))?'
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE 'basal(\\s*(anterior|posterior|medial|lateral|anteromedial))?|basalanterior|basalposterior|basalmedial|basallateral|basalanteromedial'
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE 'perih?ilar'
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE 'paracard[ií]aco'
#     --   OR LOWER(COALESCE(Laudo,'')) RLIKE 'subpleural'     
#     -- )
#     /* >>> ALTERADO: agora aceita data real OU prevista */
#     AND DataLaudoFinal IS NOT NULL
#     AND unidade IN('HOSPITAL COPA DOR', 'CLINICA SAO VICENTE', 'HOSPITAL E MATERNIDADE SAO LUIZ ANALIA FRANCO', 'HOSPITAL SAO LUIZ MORUMBI',
#                    'HOSPITAL E MATERNIDADE SAO LUIZ UNIDADE ITAIM', 'HOSPITAL QUINTA DOR', 'HOSPITAL CAXIAS DOR', 'HOSPITAL SAO LUIZ JABAQUARA',
#                    'HOSPITAL RIO BARRA', 'HOSPITAL BARRA DOR', 'HOSPITAL RIOS DOR', 'HOSPITAL ASSUNCAO', 'HOSPITAL VILLA LOBOS',
#                    'HOSPITAL E MATERNIDADE SAO LUIZ SAO CAETANO', 'HOSPITAL GLORIA DOR', 'HOSPITAL BRASIL MAUA', 'HOSPITAL BRASIL SANTO ANDRE',
#                    'HOSPITAL SANTA ISABEL', 'HOSPITAL SAO LUIZ OSASCO', 'HOSPITAL E MATERNIDADE RIBEIRAO PIRES', 'HOSPITAL BARTIRA',
#                    'HOSPITAL OESTE DOR', 'HOSPITAL CENTRAL LESTE', 'HOSPITAL NITEROI DOR', 'HOSPITAL SANTA CRUZ', 'HOSPITAL BRASIL SANTO ANDRE') 

#     -- AND idade_anos BETWEEN 30 AND 85
# ),
# -- inicialmente eu filtrava apenas a ultima (mais recente) ocorrencia de cada an
# -- por receio de perder dados alterei para a parte abaixo, mas mantem a mesma volumetria
# -- porem mantive a parte a seguir pois com ela crio a variavel an_duplicado
# -- dedup AS (
# --   SELECT *
# --   FROM (
# --     /* >>> ALTERADO: deduplica pela data final */
# --     SELECT f.*,
# --            ROW_NUMBER() OVER (PARTITION BY an ORDER BY TIMESTAMP(DataLaudoFinal) DESC) AS rn
# --     FROM filtrada f
# --   ) x
# --   WHERE rn = 1
# -- )
# -- organizo os an, nomeio o mais recente como 1 e os demais como 0
# dedup AS (
#   SELECT
#     f.*,
#     ROW_NUMBER() OVER (
#       PARTITION BY an
#       ORDER BY TIMESTAMP(DataLaudoFinal) DESC
#     ) AS rn,
#     CASE WHEN ROW_NUMBER() OVER (
#              PARTITION BY an
#              ORDER BY TIMESTAMP(DataLaudoFinal) DESC
#          ) = 1
#          THEN 1 ELSE 0 END AS an_duplicado
#   FROM filtrada f
# )
# SELECT
#   idunidade,
#   unidade,
#   id_pct,
#   paciente,
#   sexo,
#   nascimento,
#   idade_anos,
#   cpf,
#   modalidade,
#   exame,
#   dataexame,
#   tipoexame,
#   an,
#   an_duplicado,
#   idstatus,
#   -- 4 AS idstatus,                 -- mantido
#   -- idLaudo,                       
#   DataLaudoFinal AS DataLaudoLiberado,  -- >>> usa a data escolhida
#   Laudo,
#   -- current_timestamp() as datCarga
#   CAST(from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo') AS TIMESTAMP_NTZ) AS datCarga
# FROM dedup;

In [0]:
# spark.sql(f"""
# CREATE OR REPLACE TEMP VIEW {VW_BALIZADORA} AS
# WITH base AS (
#   SELECT
#       e.id_unidade                                     AS idunidade,
#       e.nme_hospital                                   AS unidade,
#       p.id_paciente                                    AS id_pct,
#       p.cli_nome                                       AS paciente,
#       p.cli_genero                                     AS sexo,
#       p.cli_data_nascimento                            AS nascimento,
#       p.doc_cpf                                        AS cpf,
#       p.cli_telefone_numero                            AS telefonePaciente,
#       p.cli_telefone_ddd                               AS telefonePacienteDDD,
#       e.tp_procedimento                                AS procedimento,
#       e.num_pedido_integracao                          AS num_pedido_integracao,
#       e.nme_regional_hospital                          AS nme_regional_hospital,
#       e.nme_convenio                                   AS nme_convenio,

#       regexp_replace(lower(coalesce(e.proced_descricao_ajustado, cast(e.cod_procedimento as string), '')), '[\\u00A0\\u2007\\u202F]', ' ') as exame_nr,

#       regexp_replace(lower(coalesce(e.dsc_codigo, '')), '[\\u00A0\\u2007\\u202F]', ' ') AS dsc_codigo_txt,
#       regexp_replace(lower(coalesce(cast(e.cod_procedimento as string), '')), '[\\u00A0\\u2007\\u202F]', ' ') AS cod_procedimento_txt,

#       /*calculo o numero de meses entre as duas datas e divido por 12 para ter o valor em anos*/
#       FLOOR(months_between(DATE(e.dt_dia_exame), DATE(p.cli_data_nascimento)) / 12) AS idade_anos,
#       CASE
#         WHEN exame_nr RLIKE '(tomografia|angio|tc|tomo)'
#           THEN 'CT'
#         ELSE 'IND'
#       END                                              AS modalidade,
#       translate(
#         regexp_replace(
#           lower(coalesce(exame_nr, cast(e.cod_procedimento as string), '')),
#           '[\\u00A0\\u2007\\u202F]', ' '
#         ),
#         'áàãâäéèêëíìîïóòõôöúùûüç',
#         'aaaaaeeeeiiiiooooouuuuc'
#       ) AS exame,

#       e.dt_dia_exame                                   AS dataexame,
#       translate(
#         regexp_replace(
#           lower(coalesce(e.nme_origem_exame, e.tp_proced_classe_encontro, '')),
#           '[\\u00A0\\u2007\\u202F]', ' '
#         ),
#         'áàãâäéèêëíìîïóòõôöúùûüç',
#         'aaaaaeeeeiiiiooooouuuuc'
#       ) AS tipoexame,

#       CAST(e.id_exame AS STRING)                       AS an,

#       e.dt_liberacao_laudo                             AS DataLaudoLiberado,
#       array_join(
#         transform(e.proced_lista_exames, x -> x.laudo_transformado),
#         '\n'
#       ) AS Laudo,
#       COALESCE(e.dt_liberacao_laudo, e.dt_previsao_laudo ) AS DataLaudoFinal,
#       CASE
#         WHEN e.dt_liberacao_laudo IS NOT NULL THEN 5
#         WHEN e.dt_previsao_laudo  IS NOT NULL THEN 4
#         ELSE 0
#       END AS idstatus

#   FROM gold_corporativo_ia.corporativo.tb_gold_mov_exame   e
#   LEFT JOIN gold_corporativo_ia.corporativo.tb_gold_mov_paciente p
#          ON p.id_paciente = e.id_patient
# ),
# filtrada AS (
#   SELECT *
#   FROM base
#   WHERE
#     UPPER(trim(procedimento)) IN  ('IMG', 'IMA')
#     AND(
#       LOWER(exame_nr) ILIKE '%tomografia%'
#       OR LOWER(exame_nr) ILIKE '%tomo%'
#       OR LOWER(exame_nr) ILIKE '%ultra%'
#       OR LOWER(exame_nr) ILIKE '%resso%'
#       OR LOWER(exame_nr) RLIKE '\\btc\\b'
#       OR LOWER(exame_nr) RLIKE '\\busg\\b'
#       OR LOWER(exame_nr) RLIKE '\\brm\\b'
#       OR LOWER(exame_nr) RLIKE '\\brnm\\b'
#       OR LOWER(dsc_codigo_txt) ILIKE '%tomografia%'
#       OR LOWER(dsc_codigo_txt) ILIKE '%tomo%'
#       OR LOWER(dsc_codigo_txt) ILIKE '%ultra%'
#       OR LOWER(dsc_codigo_txt) ILIKE '%resso%'
#       OR LOWER(dsc_codigo_txt) RLIKE '\\btc\\b'
#       OR LOWER(dsc_codigo_txt) RLIKE '\\busg\\b'
#       OR LOWER(dsc_codigo_txt) RLIKE '\\brm\\b'
#       OR LOWER(dsc_codigo_txt) RLIKE '\\brnm\\b'
#       OR LOWER(cod_procedimento_txt) ILIKE '%tomografia%'
#       OR LOWER(cod_procedimento_txt) ILIKE '%tomo%'
#       OR LOWER(cod_procedimento_txt) ILIKE '%ultra%'
#       OR LOWER(cod_procedimento_txt) ILIKE '%resso%'
#       OR LOWER(cod_procedimento_txt) RLIKE '\\btc\\b'
#       OR LOWER(cod_procedimento_txt) RLIKE '\\busg\\b'
#       OR LOWER(cod_procedimento_txt) RLIKE '\\brm\\b'
#       OR LOWER(cod_procedimento_txt) RLIKE '\\brnm\\b'

#     )
#     AND(
#       LOWER(exame_nr) ILIKE '%crânio%'
#       or LOWER(exame_nr) ILIKE '%cranio%'
#       or LOWER(exame_nr) ILIKE '%joelho%'
#       or LOWER(exame_nr) ILIKE '%articula%'
#       or LOWER(exame_nr) ILIKE '%temporomandibular%'
#       or LOWER(exame_nr) ILIKE '%face%'
#       or LOWER(exame_nr) ILIKE '%coluna%'
#       or LOWER(exame_nr) ILIKE '%lombar%'
#       or LOWER(exame_nr) ILIKE '%tornozelo%'
#       or LOWER(exame_nr) ILIKE '%pé'
#       or LOWER(exame_nr) ILIKE '%pe'
#       or LOWER(exame_nr) RLIKE '\\b(pe|pé)\\b'
#       or LOWER(exame_nr) ILIKE '%mao%'
#       or LOWER(exame_nr) ILIKE '%mão%'
#       or LOWER(exame_nr) ILIKE '%ombro%'
#       or LOWER(exame_nr) ILIKE '%punho%'
#       or LOWER(exame_nr) ILIKE '%cervical%'
#       or LOWER(exame_nr) ILIKE '%quadril%'
#       or LOWER(exame_nr) ILIKE '%quadris%'
#       or LOWER(exame_nr) ILIKE '%bacia%'
#       or LOWER(exame_nr) ILIKE '%perna%'
#       or LOWER(exame_nr) ILIKE '%braco%'
#       or LOWER(exame_nr) ILIKE '%braço%'
#       or LOWER(exame_nr) ILIKE '%cotovelo%'
#       or LOWER(exame_nr) ILIKE '%coxa%'
#       or LOWER(exame_nr) ILIKE '%intracranian%'
#       or LOWER(exame_nr) ILIKE '%pescoço%'
#       or LOWER(exame_nr) ILIKE '%pescoco%'
#       or LOWER(exame_nr) ILIKE '%cabeça%'
#       or LOWER(exame_nr) ILIKE '%cabeca%'
#       or LOWER(exame_nr) ILIKE '%dorsal%'
#       or LOWER(exame_nr) ILIKE '%osso%'
#       or LOWER(exame_nr) ILIKE '%vertebra%'
#       or LOWER(exame_nr) ILIKE '%lombossacr%'
#       or LOWER(exame_nr) ILIKE '%mastoid%'
#       or LOWER(exame_nr) ILIKE '%sacro%'
#       or LOWER(exame_nr) ILIKE '%sacra%'
#       or LOWER(dsc_codigo_txt) ILIKE '%crânio%'
#       or LOWER(dsc_codigo_txt) ILIKE '%cranio%'
#       or LOWER(dsc_codigo_txt) ILIKE '%joelho%'
#       or LOWER(dsc_codigo_txt) ILIKE '%articula%'
#       or LOWER(dsc_codigo_txt) ILIKE '%temporomandibular%'
#       or LOWER(dsc_codigo_txt) ILIKE '%face%'
#       or LOWER(dsc_codigo_txt) ILIKE '%coluna%'
#       or LOWER(dsc_codigo_txt) ILIKE '%lombar%'
#       or LOWER(dsc_codigo_txt) ILIKE '%tornozelo%'
#       or LOWER(dsc_codigo_txt) ILIKE '%pé'
#       or LOWER(dsc_codigo_txt) ILIKE '%pe'
#       or LOWER(dsc_codigo_txt) RLIKE '\\b(pe|pé)\\b'
#       or LOWER(dsc_codigo_txt) ILIKE '%mao%'
#       or LOWER(dsc_codigo_txt) ILIKE '%mão%'
#       or LOWER(dsc_codigo_txt) ILIKE '%ombro%'
#       or LOWER(dsc_codigo_txt) ILIKE '%punho%'
#       or LOWER(dsc_codigo_txt) ILIKE '%cervical%'
#       or LOWER(dsc_codigo_txt) ILIKE '%quadril%'
#       or LOWER(dsc_codigo_txt) ILIKE '%quadris%'
#       or LOWER(dsc_codigo_txt) ILIKE '%bacia%'
#       or LOWER(dsc_codigo_txt) ILIKE '%perna%'
#       or LOWER(dsc_codigo_txt) ILIKE '%braco%'
#       or LOWER(dsc_codigo_txt) ILIKE '%braço%'
#       or LOWER(dsc_codigo_txt) ILIKE '%cotovelo%'
#       or LOWER(dsc_codigo_txt) ILIKE '%coxa%'
#       or LOWER(dsc_codigo_txt) ILIKE '%intracranian%'
#       or LOWER(dsc_codigo_txt) ILIKE '%pescoço%'
#       or LOWER(dsc_codigo_txt) ILIKE '%pescoco%'
#       or LOWER(dsc_codigo_txt) ILIKE '%cabeça%'
#       or LOWER(dsc_codigo_txt) ILIKE '%cabeca%'
#       or LOWER(dsc_codigo_txt) ILIKE '%dorsal%'
#       or LOWER(dsc_codigo_txt) ILIKE '%osso%'
#       or LOWER(dsc_codigo_txt) ILIKE '%vertebra%'
#       or LOWER(dsc_codigo_txt) ILIKE '%lombossacr%'
#       or LOWER(dsc_codigo_txt) ILIKE '%mastoid%'
#       or LOWER(dsc_codigo_txt) ILIKE '%sacro%'
#       or LOWER(dsc_codigo_txt) ILIKE '%sacra%'
#       or LOWER(cod_procedimento_txt) ILIKE '%crânio%'
#       or LOWER(cod_procedimento_txt) ILIKE '%cranio%'
#       or LOWER(cod_procedimento_txt) ILIKE '%joelho%'
#       or LOWER(cod_procedimento_txt) ILIKE '%articula%'
#       or LOWER(cod_procedimento_txt) ILIKE '%temporomandibular%'
#       or LOWER(cod_procedimento_txt) ILIKE '%face%'
#       or LOWER(cod_procedimento_txt) ILIKE '%coluna%'
#       or LOWER(cod_procedimento_txt) ILIKE '%lombar%'
#       or LOWER(cod_procedimento_txt) ILIKE '%tornozelo%'
#       or LOWER(cod_procedimento_txt) ILIKE '%pé'
#       or LOWER(cod_procedimento_txt) ILIKE '%pe'
#       or LOWER(cod_procedimento_txt) RLIKE '\\b(pe|pé)\\b'
#       or LOWER(cod_procedimento_txt) ILIKE '%mao%'
#       or LOWER(cod_procedimento_txt) ILIKE '%mão%'
#       or LOWER(cod_procedimento_txt) ILIKE '%ombro%'
#       or LOWER(cod_procedimento_txt) ILIKE '%punho%'
#       or LOWER(cod_procedimento_txt) ILIKE '%cervical%'
#       or LOWER(cod_procedimento_txt) ILIKE '%quadril%'
#       or LOWER(cod_procedimento_txt) ILIKE '%quadris%'
#       or LOWER(cod_procedimento_txt) ILIKE '%bacia%'
#       or LOWER(cod_procedimento_txt) ILIKE '%perna%'
#       or LOWER(cod_procedimento_txt) ILIKE '%braco%'
#       or LOWER(cod_procedimento_txt) ILIKE '%braço%'
#       or LOWER(cod_procedimento_txt) ILIKE '%cotovelo%'
#       or LOWER(cod_procedimento_txt) ILIKE '%coxa%'
#       or LOWER(cod_procedimento_txt) ILIKE '%intracranian%'
#       or LOWER(cod_procedimento_txt) ILIKE '%pescoço%'
#       or LOWER(cod_procedimento_txt) ILIKE '%pescoco%'
#       or LOWER(cod_procedimento_txt) ILIKE '%cabeça%'
#       or LOWER(cod_procedimento_txt) ILIKE '%cabeca%'
#       or LOWER(cod_procedimento_txt) ILIKE '%dorsal%'
#       or LOWER(cod_procedimento_txt) ILIKE '%osso%'
#       or LOWER(cod_procedimento_txt) ILIKE '%vertebra%'
#       or LOWER(cod_procedimento_txt) ILIKE '%lombossacr%'
#       or LOWER(cod_procedimento_txt) ILIKE '%mastoid%'
#       or LOWER(cod_procedimento_txt) ILIKE '%sacro%'
#       or LOWER(cod_procedimento_txt) ILIKE '%sacra%'

#     )
#     AND(
#       LOWER(exame_nr) NOT LIKE '%angio%' 
#       AND LOWER(exame_nr) NOT LIKE '%artérias%' 
#       AND LOWER(exame_nr) NOT LIKE '%carótidas%' 
#       AND LOWER(exame_nr) NOT LIKE '%arterias%' 
#       AND LOWER(exame_nr) NOT LIKE '%carotidas%' 
#       AND LOWER(exame_nr) NOT LIKE '%venos%' 
#       AND LOWER(exame_nr) NOT LIKE '%paaf%' 
#       AND LOWER(exame_nr) NOT LIKE '%punção%'
#       AND LOWER(exame_nr) NOT LIKE '%biop%'
#       AND LOWER(exame_nr) NOT LIKE '%bióp%'
#       AND LOWER(dsc_codigo_txt) NOT LIKE '%angio%' 
#       AND LOWER(dsc_codigo_txt) NOT LIKE '%artérias%' 
#       AND LOWER(dsc_codigo_txt) NOT LIKE '%carótidas%' 
#       AND LOWER(dsc_codigo_txt) NOT LIKE '%arterias%' 
#       AND LOWER(dsc_codigo_txt) NOT LIKE '%carotidas%' 
#       AND LOWER(dsc_codigo_txt) NOT LIKE '%venos%' 
#       AND LOWER(dsc_codigo_txt) NOT LIKE '%paaf%' 
#       AND LOWER(dsc_codigo_txt) NOT LIKE '%punção%'
#       AND LOWER(dsc_codigo_txt) NOT LIKE '%biop%'
#       AND LOWER(dsc_codigo_txt) NOT LIKE '%bióp%'
#       AND LOWER(cod_procedimento_txt) NOT LIKE '%angio%' 
#       AND LOWER(cod_procedimento_txt) NOT LIKE '%artérias%' 
#       AND LOWER(cod_procedimento_txt) NOT LIKE '%carótidas%' 
#       AND LOWER(cod_procedimento_txt) NOT LIKE '%arterias%' 
#       AND LOWER(cod_procedimento_txt) NOT LIKE '%carotidas%' 
#       AND LOWER(cod_procedimento_txt) NOT LIKE '%venos%' 
#       AND LOWER(cod_procedimento_txt) NOT LIKE '%paaf%' 
#       AND LOWER(cod_procedimento_txt) NOT LIKE '%punção%'
#       AND LOWER(cod_procedimento_txt) NOT LIKE '%biop%'
#       AND LOWER(cod_procedimento_txt) NOT LIKE '%bióp%'
#     )  
#     AND DataLaudoFinal IS NOT NULL
#     AND exame != ""
#     AND DATE(DataLaudoFinal) BETWEEN date_sub(current_date(), 30)
#                                  AND date_sub(current_date(), 1) -- ontem
# ),

# dedup AS (
#   SELECT
#     f.*,
#     ROW_NUMBER() OVER (
#       PARTITION BY an
#       ORDER BY TIMESTAMP(DataLaudoFinal) DESC
#     ) AS rn,
#     CASE WHEN ROW_NUMBER() OVER (
#              PARTITION BY an
#              ORDER BY TIMESTAMP(DataLaudoFinal) DESC
#          ) = 1
#          THEN 1 ELSE 0 END AS an_duplicado
#   FROM filtrada f
# )
# SELECT
#   idunidade,
#   unidade,
#   id_pct,
#   paciente,
#   telefonePacienteDDD,
#   telefonePaciente,
#   sexo,
#   nascimento,
#   idade_anos,
#   cpf,
#   modalidade,
#   exame,
#   num_pedido_integracao,
#   nme_regional_hospital,
#   nme_convenio,
#   dataexame,
#   tipoexame,
#   an,
#   an_duplicado,
#   idstatus,
#   -- 4 AS idstatus,                 -- mantido
#   -- idLaudo,                       
#   DataLaudoFinal AS DataLaudoLiberado,  -- >>> usa a data escolhida
#   Laudo,
#   -- current_timestamp() as datCarga
#   CAST(from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo') AS TIMESTAMP_NTZ) AS dataExecucaoModelo
# FROM dedup;
# """)

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW {VW_BALIZADORA} AS
WITH base AS (
  SELECT
      e.id_unidade                                     AS idunidade,
      e.nme_hospital                                   AS unidade,
      p.id_paciente                                    AS id_pct,
      p.cli_nome                                       AS paciente,
      p.cli_genero                                     AS sexo,
      p.cli_data_nascimento                            AS nascimento,
      p.doc_cpf                                        AS cpf,
      p.cli_telefone_numero                            AS telefonePaciente,
      p.cli_telefone_ddd                               AS telefonePacienteDDD,
      e.tp_procedimento                                AS procedimento,
      e.num_pedido_integracao                          AS num_pedido_integracao,
      e.nme_regional_hospital                          AS nme_regional_hospital,
      e.nme_convenio                                   AS nme_convenio,

      regexp_replace(lower(coalesce(e.proced_descricao_ajustado, cast(e.cod_procedimento as string), '')), '[\\u00A0\\u2007\\u202F]', ' ') as exame_nr,

      regexp_replace(lower(coalesce(e.dsc_codigo, '')), '[\\u00A0\\u2007\\u202F]', ' ') AS dsc_codigo_txt,
      regexp_replace(lower(coalesce(cast(e.cod_procedimento as string), '')), '[\\u00A0\\u2007\\u202F]', ' ') AS cod_procedimento_txt,

      /*calculo o numero de meses entre as duas datas e divido por 12 para ter o valor em anos*/
      FLOOR(months_between(DATE(e.dt_dia_exame), DATE(p.cli_data_nascimento)) / 12) AS idade_anos,
      CASE
        WHEN exame_nr RLIKE '(tomografia|angio|tc|tomo)'
          THEN 'CT'
        ELSE 'IND'
      END                                              AS modalidade,
      translate(
        regexp_replace(
          lower(coalesce(exame_nr, cast(e.cod_procedimento as string), '')),
          '[\\u00A0\\u2007\\u202F]', ' '
        ),
        'áàãâäéèêëíìîïóòõôöúùûüç',
        'aaaaaeeeeiiiiooooouuuuc'
      ) AS exame,

      e.dt_dia_exame                                   AS dataexame,
      translate(
        regexp_replace(
          lower(coalesce(e.nme_origem_exame, e.tp_proced_classe_encontro, '')),
          '[\\u00A0\\u2007\\u202F]', ' '
        ),
        'áàãâäéèêëíìîïóòõôöúùûüç',
        'aaaaaeeeeiiiiooooouuuuc'
      ) AS tipoexame,

      CAST(e.id_exame AS STRING)                       AS an,

      e.dt_liberacao_laudo                             AS DataLaudoLiberado,
      array_join(
        transform(e.proced_lista_exames, x -> x.laudo_transformado),
        '\n'
      ) AS Laudo,
      COALESCE(e.dt_liberacao_laudo, e.dt_previsao_laudo ) AS DataLaudoFinal,
      CASE
        WHEN e.dt_liberacao_laudo IS NOT NULL THEN 5
        WHEN e.dt_previsao_laudo  IS NOT NULL THEN 4
        ELSE 0
      END AS idstatus

  FROM gold_corporativo_ia.corporativo.tb_gold_mov_exame   e
  LEFT JOIN gold_corporativo_ia.corporativo.tb_gold_mov_paciente p
         ON p.id_paciente = e.id_patient
),
filtrada AS (
  SELECT *
  FROM base
  WHERE
    UPPER(trim(procedimento)) IN  ('IMG', 'IMA')
    AND(
      LOWER(exame_nr) ILIKE '%tomografia%'
      OR LOWER(exame_nr) ILIKE '%tomo%'
      OR LOWER(exame_nr) ILIKE '%tc%'
      OR LOWER(exame_nr) ILIKE '%ultra%'
      OR LOWER(exame_nr) ILIKE '%resso%'
      OR LOWER(exame_nr) RLIKE '\\btc\\b'
      OR LOWER(exame_nr) RLIKE '\\busg\\b'
      OR LOWER(exame_nr) RLIKE '\\brm\\b'
      OR LOWER(exame_nr) RLIKE '\\brnm\\b'
      OR LOWER(dsc_codigo_txt) ILIKE '%tomografia%'
      OR LOWER(dsc_codigo_txt) ILIKE '%tomo%'
      OR LOWER(dsc_codigo_txt) ILIKE '%tc%'
      OR LOWER(dsc_codigo_txt) ILIKE '%ultra%'
      OR LOWER(dsc_codigo_txt) ILIKE '%resso%'
      OR LOWER(dsc_codigo_txt) RLIKE '\\btc\\b'
      OR LOWER(dsc_codigo_txt) RLIKE '\\busg\\b'
      OR LOWER(dsc_codigo_txt) RLIKE '\\brm\\b'
      OR LOWER(dsc_codigo_txt) RLIKE '\\brnm\\b'
      OR LOWER(cod_procedimento_txt) ILIKE '%tomografia%'
      OR LOWER(cod_procedimento_txt) ILIKE '%tomo%'
      OR LOWER(cod_procedimento_txt) ILIKE '%tc%'
      OR LOWER(cod_procedimento_txt) ILIKE '%ultra%'
      OR LOWER(cod_procedimento_txt) ILIKE '%resso%'
      OR LOWER(cod_procedimento_txt) RLIKE '\\btc\\b'
      OR LOWER(cod_procedimento_txt) RLIKE '\\busg\\b'
      OR LOWER(cod_procedimento_txt) RLIKE '\\brm\\b'
      OR LOWER(cod_procedimento_txt) RLIKE '\\brnm\\b'

    )
    AND(
      LOWER(exame_nr) ILIKE '%crânio%'
      or LOWER(exame_nr) ILIKE '%cranio%'
      or LOWER(exame_nr) ILIKE '%joelho%'
      or LOWER(exame_nr) ILIKE '%articula%'
      or LOWER(exame_nr) ILIKE '%temporomandibular%'
      or LOWER(exame_nr) ILIKE '%face%'
      or LOWER(exame_nr) ILIKE '%coluna%'
      or LOWER(exame_nr) ILIKE '%lombar%'
      or LOWER(exame_nr) ILIKE '%tornozelo%'
      or LOWER(exame_nr) ILIKE '%pé'
      or LOWER(exame_nr) ILIKE '%pe'
      or LOWER(exame_nr) RLIKE '\\b(pe|pé)\\b'
      or LOWER(exame_nr) ILIKE '%mao%'
      or LOWER(exame_nr) ILIKE '%mão%'
      or LOWER(exame_nr) ILIKE '%ombro%'
      or LOWER(exame_nr) ILIKE '%punho%'
      or LOWER(exame_nr) ILIKE '%cervical%'
      or LOWER(exame_nr) ILIKE '%quadril%'
      or LOWER(exame_nr) ILIKE '%quadris%'
      or LOWER(exame_nr) ILIKE '%bacia%'
      or LOWER(exame_nr) ILIKE '%perna%'
      or LOWER(exame_nr) ILIKE '%braco%'
      or LOWER(exame_nr) ILIKE '%braço%'
      or LOWER(exame_nr) ILIKE '%cotovelo%'
      or LOWER(exame_nr) ILIKE '%coxa%'
      or LOWER(exame_nr) ILIKE '%intracranian%'
      or LOWER(exame_nr) ILIKE '%pescoço%'
      or LOWER(exame_nr) ILIKE '%pescoco%'
      or LOWER(exame_nr) ILIKE '%cabeça%'
      or LOWER(exame_nr) ILIKE '%cabeca%'
      or LOWER(exame_nr) ILIKE '%dorsal%'
      or LOWER(exame_nr) ILIKE '%osso%'
      or LOWER(exame_nr) ILIKE '%vertebra%'
      or LOWER(exame_nr) ILIKE '%lombossacr%'
      or LOWER(exame_nr) ILIKE '%mastoid%'
      or LOWER(exame_nr) ILIKE '%sacro%'
      or LOWER(exame_nr) ILIKE '%sacra%'
      or LOWER(dsc_codigo_txt) ILIKE '%crânio%'
      or LOWER(dsc_codigo_txt) ILIKE '%cranio%'
      or LOWER(dsc_codigo_txt) ILIKE '%joelho%'
      or LOWER(dsc_codigo_txt) ILIKE '%articula%'
      or LOWER(dsc_codigo_txt) ILIKE '%temporomandibular%'
      or LOWER(dsc_codigo_txt) ILIKE '%face%'
      or LOWER(dsc_codigo_txt) ILIKE '%coluna%'
      or LOWER(dsc_codigo_txt) ILIKE '%lombar%'
      or LOWER(dsc_codigo_txt) ILIKE '%tornozelo%'
      or LOWER(dsc_codigo_txt) ILIKE '%pé'
      or LOWER(dsc_codigo_txt) ILIKE '%pe'
      or LOWER(dsc_codigo_txt) RLIKE '\\b(pe|pé)\\b'
      or LOWER(dsc_codigo_txt) ILIKE '%mao%'
      or LOWER(dsc_codigo_txt) ILIKE '%mão%'
      or LOWER(dsc_codigo_txt) ILIKE '%ombro%'
      or LOWER(dsc_codigo_txt) ILIKE '%punho%'
      or LOWER(dsc_codigo_txt) ILIKE '%cervical%'
      or LOWER(dsc_codigo_txt) ILIKE '%quadril%'
      or LOWER(dsc_codigo_txt) ILIKE '%quadris%'
      or LOWER(dsc_codigo_txt) ILIKE '%bacia%'
      or LOWER(dsc_codigo_txt) ILIKE '%perna%'
      or LOWER(dsc_codigo_txt) ILIKE '%braco%'
      or LOWER(dsc_codigo_txt) ILIKE '%braço%'
      or LOWER(dsc_codigo_txt) ILIKE '%cotovelo%'
      or LOWER(dsc_codigo_txt) ILIKE '%coxa%'
      or LOWER(dsc_codigo_txt) ILIKE '%intracranian%'
      or LOWER(dsc_codigo_txt) ILIKE '%pescoço%'
      or LOWER(dsc_codigo_txt) ILIKE '%pescoco%'
      or LOWER(dsc_codigo_txt) ILIKE '%cabeça%'
      or LOWER(dsc_codigo_txt) ILIKE '%cabeca%'
      or LOWER(dsc_codigo_txt) ILIKE '%dorsal%'
      or LOWER(dsc_codigo_txt) ILIKE '%osso%'
      or LOWER(dsc_codigo_txt) ILIKE '%vertebra%'
      or LOWER(dsc_codigo_txt) ILIKE '%lombossacr%'
      or LOWER(dsc_codigo_txt) ILIKE '%mastoid%'
      or LOWER(dsc_codigo_txt) ILIKE '%sacro%'
      or LOWER(dsc_codigo_txt) ILIKE '%sacra%'
      or LOWER(cod_procedimento_txt) ILIKE '%crânio%'
      or LOWER(cod_procedimento_txt) ILIKE '%cranio%'
      or LOWER(cod_procedimento_txt) ILIKE '%joelho%'
      or LOWER(cod_procedimento_txt) ILIKE '%articula%'
      or LOWER(cod_procedimento_txt) ILIKE '%temporomandibular%'
      or LOWER(cod_procedimento_txt) ILIKE '%face%'
      or LOWER(cod_procedimento_txt) ILIKE '%coluna%'
      or LOWER(cod_procedimento_txt) ILIKE '%lombar%'
      or LOWER(cod_procedimento_txt) ILIKE '%tornozelo%'
      or LOWER(cod_procedimento_txt) ILIKE '%pé'
      or LOWER(cod_procedimento_txt) ILIKE '%pe'
      or LOWER(cod_procedimento_txt) RLIKE '\\b(pe|pé)\\b'
      or LOWER(cod_procedimento_txt) ILIKE '%mao%'
      or LOWER(cod_procedimento_txt) ILIKE '%mão%'
      or LOWER(cod_procedimento_txt) ILIKE '%ombro%'
      or LOWER(cod_procedimento_txt) ILIKE '%punho%'
      or LOWER(cod_procedimento_txt) ILIKE '%cervical%'
      or LOWER(cod_procedimento_txt) ILIKE '%quadril%'
      or LOWER(cod_procedimento_txt) ILIKE '%quadris%'
      or LOWER(cod_procedimento_txt) ILIKE '%bacia%'
      or LOWER(cod_procedimento_txt) ILIKE '%perna%'
      or LOWER(cod_procedimento_txt) ILIKE '%braco%'
      or LOWER(cod_procedimento_txt) ILIKE '%braço%'
      or LOWER(cod_procedimento_txt) ILIKE '%cotovelo%'
      or LOWER(cod_procedimento_txt) ILIKE '%coxa%'
      or LOWER(cod_procedimento_txt) ILIKE '%intracranian%'
      or LOWER(cod_procedimento_txt) ILIKE '%pescoço%'
      or LOWER(cod_procedimento_txt) ILIKE '%pescoco%'
      or LOWER(cod_procedimento_txt) ILIKE '%cabeça%'
      or LOWER(cod_procedimento_txt) ILIKE '%cabeca%'
      or LOWER(cod_procedimento_txt) ILIKE '%dorsal%'
      or LOWER(cod_procedimento_txt) ILIKE '%osso%'
      or LOWER(cod_procedimento_txt) ILIKE '%vertebra%'
      or LOWER(cod_procedimento_txt) ILIKE '%lombossacr%'
      or LOWER(cod_procedimento_txt) ILIKE '%mastoid%'
      or LOWER(cod_procedimento_txt) ILIKE '%sacro%'
      or LOWER(cod_procedimento_txt) ILIKE '%sacra%'

    )
    AND(
      LOWER(COALESCE(exame_nr, '')) NOT LIKE '%angio%' 
      AND LOWER(COALESCE(exame_nr, '')) NOT LIKE '%artérias%' 
      AND LOWER(COALESCE(exame_nr, '')) NOT LIKE '%carótidas%' 
      AND LOWER(COALESCE(exame_nr, '')) NOT LIKE '%arterias%' 
      AND LOWER(COALESCE(exame_nr, '')) NOT LIKE '%carotidas%' 
      AND LOWER(COALESCE(exame_nr, '')) NOT LIKE '%venos%' 
      AND LOWER(COALESCE(exame_nr, '')) NOT LIKE '%paaf%' 
      AND LOWER(COALESCE(exame_nr, '')) NOT LIKE '%punção%'
      AND LOWER(COALESCE(exame_nr, '')) NOT LIKE '%biop%'
      AND LOWER(COALESCE(exame_nr, '')) NOT LIKE '%bióp%'
      AND LOWER(COALESCE(dsc_codigo_txt, '')) NOT LIKE '%angio%' 
      AND LOWER(COALESCE(dsc_codigo_txt, '')) NOT LIKE '%artérias%' 
      AND LOWER(COALESCE(dsc_codigo_txt, '')) NOT LIKE '%carótidas%' 
      AND LOWER(COALESCE(dsc_codigo_txt, '')) NOT LIKE '%arterias%' 
      AND LOWER(COALESCE(dsc_codigo_txt, '')) NOT LIKE '%carotidas%' 
      AND LOWER(COALESCE(dsc_codigo_txt, '')) NOT LIKE '%venos%' 
      AND LOWER(COALESCE(dsc_codigo_txt, '')) NOT LIKE '%paaf%' 
      AND LOWER(COALESCE(dsc_codigo_txt, '')) NOT LIKE '%punção%'
      AND LOWER(COALESCE(dsc_codigo_txt, '')) NOT LIKE '%biop%'
      AND LOWER(COALESCE(dsc_codigo_txt, '')) NOT LIKE '%bióp%'
      AND LOWER(COALESCE(cod_procedimento_txt, '')) NOT LIKE '%angio%' 
      AND LOWER(COALESCE(cod_procedimento_txt, '')) NOT LIKE '%artérias%' 
      AND LOWER(COALESCE(cod_procedimento_txt, '')) NOT LIKE '%carótidas%' 
      AND LOWER(COALESCE(cod_procedimento_txt, '')) NOT LIKE '%arterias%' 
      AND LOWER(COALESCE(cod_procedimento_txt, '')) NOT LIKE '%carotidas%' 
      AND LOWER(COALESCE(cod_procedimento_txt, '')) NOT LIKE '%venos%' 
      AND LOWER(COALESCE(cod_procedimento_txt, '')) NOT LIKE '%paaf%' 
      AND LOWER(COALESCE(cod_procedimento_txt, '')) NOT LIKE '%punção%'
      AND LOWER(COALESCE(cod_procedimento_txt, '')) NOT LIKE '%biop%'
      AND LOWER(COALESCE(cod_procedimento_txt, '')) NOT LIKE '%bióp%'

    )  
    AND DataLaudoFinal IS NOT NULL
    AND exame != ""
    AND DATE(DataLaudoFinal) BETWEEN date_sub(current_date(), 30)
                                 AND date_sub(current_date(), 1) -- ontem
),

dedup AS (
  SELECT
    f.*,
    ROW_NUMBER() OVER (
      PARTITION BY an
      ORDER BY TIMESTAMP(DataLaudoFinal) DESC
    ) AS rn,
    CASE WHEN ROW_NUMBER() OVER (
             PARTITION BY an
             ORDER BY TIMESTAMP(DataLaudoFinal) DESC
         ) = 1
         THEN 1 ELSE 0 END AS an_duplicado
  FROM filtrada f
)
SELECT
  idunidade,
  unidade,
  id_pct,
  paciente,
  telefonePacienteDDD,
  telefonePaciente,
  sexo,
  nascimento,
  idade_anos,
  cpf,
  modalidade,
  exame,
  num_pedido_integracao,
  nme_regional_hospital,
  nme_convenio,
  dataexame,
  tipoexame,
  an,
  an_duplicado,
  idstatus,
  -- 4 AS idstatus,                 -- mantido
  -- idLaudo,                       
  DataLaudoFinal AS DataLaudoLiberado,  -- >>> usa a data escolhida
  Laudo,
  -- current_timestamp() as datCarga
  CAST(from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo') AS TIMESTAMP_NTZ) AS dataExecucaoModelo
FROM dedup;
""")

In [0]:
spark.sql(f"""CREATE TABLE IF NOT EXISTS {SCHEMA}.{OUTPUT_TABLENAME}
            USING DELTA
            AS
            SELECT *
            FROM {VW_BALIZADORA}
            WHERE 1 = 0
            """)

In [0]:
# spark.sql(f"""
# ALTER TABLE {SCHEMA}.{OUTPUT_TABLENAME}
# ADD COLUMNS (
#   num_pedido_integracao STRING,
#   nme_regional_hospital STRING,
#   nme_convenio STRING)
# """)

In [0]:
spark.sql(f"""
INSERT INTO {SCHEMA}.{OUTPUT_TABLENAME} (
    idunidade,
    unidade,
    id_pct,
    paciente,
    telefonePacienteDDD,
    telefonePaciente,
    sexo,
    nascimento,
    idade_anos,
    cpf,
    modalidade,
    exame,
    num_pedido_integracao,
    nme_regional_hospital,
    nme_convenio,
    dataexame,
    tipoexame,
    an,
    an_duplicado,
    idstatus,
    DataLaudoLiberado,
    Laudo,
    dataExecucaoModelo
)
SELECT
    s.idunidade,
    s.unidade,
    s.id_pct,
    s.paciente,
    s.telefonePacienteDDD,
    s.telefonePaciente,
    s.sexo,
    s.nascimento,
    s.idade_anos,
    s.cpf,
    s.modalidade,
    s.exame,
    s.num_pedido_integracao,
    s.nme_regional_hospital,
    s.nme_convenio,
    s.dataexame,              
    s.tipoexame,
    s.an,
    s.an_duplicado,
    s.idstatus,
    s.DataLaudoLiberado,
    s.Laudo,
    s.dataExecucaoModelo
FROM {VW_BALIZADORA} s
LEFT ANTI JOIN ia.{OUTPUT_TABLENAME} t
    ON t.an        = s.an
""")

In [0]:
# spark.sql(f"""INSERT INTO {CATALOG}.{OUTPUT_TABLENAME}
#               SELECT s.*
#               FROM {VW_BALIZADORA} s
#               LEFT ANTI JOIN ia.{OUTPUT_TABLENAME} t
#                 ON  t.idunidade = s.idunidade
#                 AND t.id_pct = s.id_pct
#                 AND t.exame = s.exame
#                 AND t.dataexame = s.dataexame
#                 AND t.tipoexame = s.tipoexame
#                 AND t.an = s.an
#             """)

####Consultas

In [0]:
spark.sql(f"""
            select *
            from {SCHEMA}.{OUTPUT_TABLENAME}
            where cast(dataExecucaoModelo as date) = current_date()
            """).display()

In [0]:
spark.sql(f"""
            SELECT
                dataExecucaoModelo,
                COUNT(*) AS total_registros
            FROM {SCHEMA}.{OUTPUT_TABLENAME}
            GROUP BY dataExecucaoModelo
            ORDER BY dataExecucaoModelo
        """).display()

In [0]:
spark.sql(f"""
SELECT
  CAST(DataLaudoLiberado AS DATE) AS data_exame,
  unidade,
  COUNT(*) AS total_exames
FROM {SCHEMA}.{OUTPUT_TABLENAME}
GROUP BY
  CAST(DataLaudoLiberado AS DATE),
  unidade
ORDER BY
  data_exame DESC,
  unidade
""").display()

In [0]:
dbutils.notebook.exit("Final Notebook")

In [0]:
# %sql
# select
# --exam.id_exame,
# --exam.regional,
# --exam.unidade,
# --exam.data_laudo,
# distinct exam.descricao_laudo
# --exam.exploded.laudo_transformado as laudo
# from
# (SELECT
#   id_exame,
#   proced_descricao as descricao_laudo,
#   explode(proced_lista_exames) as exploded,
#   dt_liberacao_laudo as data_laudo,
#   emp_descricao_unidade as unidade,
#   nme_regional_hospital as regional
# FROM gold_corporativo_ia.corporativo.tb_gold_mov_exame
# WHERE 
#   (lower(proced_descricao) ILIKE '%tomografia' or lower(proced_descricao) ILIKE '%ultra%' or lower(proced_descricao) ILIKE '%resso%'
#   or lower(proced_descricao) ILIKE '%tc' or lower(proced_descricao) ILIKE '%usg' or lower(proced_descricao) ILIKE '%rm')
#   and (LOWER(proced_descricao) ILIKE '%crânio%' or LOWER(proced_descricao) ILIKE '%cranio%' or LOWER(proced_descricao) ILIKE '%joelho%' or LOWER(proced_descricao) ILIKE '%articula%' or LOWER(proced_descricao) ILIKE '%temporomandibular%' or LOWER(proced_descricao) ILIKE '%face%' or LOWER(proced_descricao) ILIKE '%coluna%' or LOWER(proced_descricao) ILIKE '%lombar%' or LOWER(proced_descricao) ILIKE '%tornozelo%' or LOWER(proced_descricao) ILIKE '%pé' or LOWER(proced_descricao) ILIKE '%pe' or LOWER(proced_descricao) ILIKE '%mao%' or LOWER(proced_descricao) ILIKE '%mão%' or LOWER(proced_descricao) ILIKE '%ombro%' or LOWER(proced_descricao) ILIKE '%punho%' or LOWER(proced_descricao) ILIKE '%cervical%' or LOWER(proced_descricao) ILIKE '%quadril%' or LOWER(proced_descricao) ILIKE '%quadris%' or LOWER(proced_descricao) ILIKE '%bacia%' or LOWER(proced_descricao) ILIKE '%perna%' or LOWER(proced_descricao) ILIKE '%braco%' or LOWER(proced_descricao) ILIKE '%braço%' or LOWER(proced_descricao) ILIKE '%cotovelo%' or LOWER(proced_descricao) ILIKE '%coxa%' or LOWER(proced_descricao) ILIKE '%intracranian%' or LOWER(proced_descricao) ILIKE '%pescoço%' or LOWER(proced_descricao) ILIKE '%pescoco%' or LOWER(proced_descricao) ILIKE '%cabeça%' or LOWER(proced_descricao) ILIKE '%cabeca%' or LOWER(proced_descricao) ILIKE '%dorsal%' or LOWER(proced_descricao) ILIKE '%osso%' or LOWER(proced_descricao) ILIKE '%vertebra%' or LOWER(proced_descricao) ILIKE '%lombossacr%' or LOWER(proced_descricao) ILIKE '%mastoid%' or LOWER(proced_descricao) ILIKE '%sacro%' or LOWER(proced_descricao) ILIKE '%sacra%')
#   and (LOWER(proced_descricao) NOT LIKE '%angio%' AND LOWER(proced_descricao) NOT LIKE '%artérias%' AND LOWER(proced_descricao) NOT LIKE '%carótidas%' AND LOWER(proced_descricao) NOT LIKE '%arterias%' AND LOWER(proced_descricao) NOT LIKE '%carotidas%' AND LOWER(proced_descricao) NOT LIKE '%venos%' AND LOWER(proced_descricao) NOT LIKE '%paaf%' AND LOWER(proced_descricao) NOT LIKE '%punção%'AND LOWER(proced_descricao) NOT LIKE '%biop%'AND LOWER(proced_descricao) NOT LIKE '%bióp%')) as exam
# WHERE exam.exploded.laudo_transformado IS NOT NULL
# AND exam.exploded.laudo_transformado != ""
# AND exam.data_laudo between '2025-11-01' and '2025-11-30'
# order by 1;

In [0]:
# %sql
# select
# --exam.id_exame,
# --exam.regional,
# --exam.unidade,
# --exam.data_laudo,
# distinct exam.descricao_laudo,
# count(*) as qt_laudos
# --exam.exploded.laudo_transformado as laudo
# from
# (SELECT
#   id_exame,
#   proced_descricao as descricao_laudo,
#   explode(proced_lista_exames) as exploded,
#   dt_liberacao_laudo as data_laudo,
#   emp_descricao_unidade as unidade,
#   nme_regional_hospital as regional
# FROM gold_corporativo_ia.corporativo.tb_gold_mov_exame
# WHERE 
#   (lower(proced_descricao) ILIKE '%tomografia' or lower(proced_descricao) ILIKE '%ultra%' or lower(proced_descricao) ILIKE '%resso%'
#   or lower(proced_descricao) ILIKE '%tc' or lower(proced_descricao) ILIKE '%usg' or lower(proced_descricao) ILIKE '%rm')
#   and (LOWER(proced_descricao) ILIKE '%crânio%' or LOWER(proced_descricao) ILIKE '%cranio%' or LOWER(proced_descricao) ILIKE '%joelho%' or LOWER(proced_descricao) ILIKE '%articula%' or LOWER(proced_descricao) ILIKE '%temporomandibular%' or LOWER(proced_descricao) ILIKE '%face%' or LOWER(proced_descricao) ILIKE '%coluna%' or LOWER(proced_descricao) ILIKE '%lombar%' or LOWER(proced_descricao) ILIKE '%tornozelo%' or LOWER(proced_descricao) ILIKE '%pé' or LOWER(proced_descricao) ILIKE '%pe' or LOWER(proced_descricao) ILIKE '%mao%' or LOWER(proced_descricao) ILIKE '%mão%' or LOWER(proced_descricao) ILIKE '%ombro%' or LOWER(proced_descricao) ILIKE '%punho%' or LOWER(proced_descricao) ILIKE '%cervical%' or LOWER(proced_descricao) ILIKE '%quadril%' or LOWER(proced_descricao) ILIKE '%quadris%' or LOWER(proced_descricao) ILIKE '%bacia%' or LOWER(proced_descricao) ILIKE '%perna%' or LOWER(proced_descricao) ILIKE '%braco%' or LOWER(proced_descricao) ILIKE '%braço%' or LOWER(proced_descricao) ILIKE '%cotovelo%' or LOWER(proced_descricao) ILIKE '%coxa%' or LOWER(proced_descricao) ILIKE '%intracranian%' or LOWER(proced_descricao) ILIKE '%pescoço%' or LOWER(proced_descricao) ILIKE '%pescoco%' or LOWER(proced_descricao) ILIKE '%cabeça%' or LOWER(proced_descricao) ILIKE '%cabeca%' or LOWER(proced_descricao) ILIKE '%dorsal%' or LOWER(proced_descricao) ILIKE '%osso%' or LOWER(proced_descricao) ILIKE '%vertebra%' or LOWER(proced_descricao) ILIKE '%lombossacr%' or LOWER(proced_descricao) ILIKE '%mastoid%' or LOWER(proced_descricao) ILIKE '%sacro%' or LOWER(proced_descricao) ILIKE '%sacra%')
#   and (LOWER(proced_descricao) NOT LIKE '%angio%' AND LOWER(proced_descricao) NOT LIKE '%artérias%' AND LOWER(proced_descricao) NOT LIKE '%carótidas%' AND LOWER(proced_descricao) NOT LIKE '%arterias%' AND LOWER(proced_descricao) NOT LIKE '%carotidas%' AND LOWER(proced_descricao) NOT LIKE '%venos%' AND LOWER(proced_descricao) NOT LIKE '%paaf%' AND LOWER(proced_descricao) NOT LIKE '%punção%'AND LOWER(proced_descricao) NOT LIKE '%biop%'AND LOWER(proced_descricao) NOT LIKE '%bióp%')) as exam
# WHERE exam.exploded.laudo_transformado IS NOT NULL
# AND exam.exploded.laudo_transformado != ""
# AND exam.data_laudo between '2025-11-01' and '2025-11-30'
# group by descricao_laudo
# order by 2 desc;

In [0]:
# %sql
# select *
# from ia.dev_tb_diamond_mod_reumatologia_saida
# where cast(dataExecucaoModelo as date) = '2026-01-22'

In [0]:
# %sql
# delete from ia.dev_tb_diamond_mod_reumatologia
# where cast(dataExecucaoModelo as date) = '2026-01-23'


In [0]:
# %sql
# select *
# from ia.dev_tb_diamond_mod_pulmao_saida_salesforce
# where dataExecucaoModelo = '2026-01-23'
# limit 100